In [ ]:
# Install required libraries
!pip install -q langchain langchain-openai langchain-core pydantic

import os
import json
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

print("--- API Setup ---")
# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

# Mandatory LangSmith Tracing Setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith API Key: ")
os.environ["LANGCHAIN_PROJECT"] = "GenAI_Resume_Screening"

# Initialize the LLM (using a standard, cost-effective model)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 8.8 MB/s eta 0:00:00
--- API Setup ---
Enter OpenAI API Key: ··········
Enter LangSmith API Key: ··········


In [4]:
# Install required libraries (Swapped HuggingFace for Google GenAI)
!pip install -q langchain langchain-google-genai langchain-core pydantic

import os
from getpass import getpass
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import JsonOutputParser

print("--- API Setup ---")
# Enter your key from https://aistudio.google.com/app/apikey
os.environ["GOOGLE_API_KEY"] = getpass("Enter Google Gemini API Key: ")

# Mandatory LangSmith Tracing Setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith API Key: ")
os.environ["LANGCHAIN_PROJECT"] = "GenAI_Resume_Screening"

# Initialize Gemini (Super fast and built for complex JSON extraction)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

print("\n--- Building Resume Screening Pipeline ---")

# Step 1: Skill Extraction Chain
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
    You are an expert technical recruiter. Extract information from this resume.
    CRITICAL RULE: Do NOT assume or hallucinate skills not explicitly present.

    Resume Text:
    {resume}

    Return ONLY a raw JSON object with keys: "skills" (list), "experience_years" (string), "tools" (list).
    """
)
extract_chain = extract_prompt | llm | JsonOutputParser()

# Step 2: Scoring & Explanation Chain
score_prompt = PromptTemplate(
    input_variables=["extracted_data", "job_description"],
    template="""
    Evaluate the candidate based ONLY on the extracted resume data and job description.

    Job Description:
    {job_description}

    Extracted Candidate Data:
    {extracted_data}

    Provide a fit score (0-100) and explanation.
    Return ONLY a raw JSON object with keys: "score" (integer), "explanation" (string).
    """
)
score_chain = score_prompt | llm | JsonOutputParser()

# Step 3: LCEL Unified Pipeline
full_screening_pipeline = (
    {
        "extracted_data": {"resume": lambda x: x["resume"]} | extract_chain,
        "job_description": lambda x: x["job_description"]
    }
    | score_chain
)

# Step 4: Run Candidates
job_desc = """
Looking for a Data Scientist with 3+ years experience in Python, Machine Learning, and SQL.
Tools: Git, Docker, AWS.
"""

resumes = {
    "Strong Candidate": "Jane Doe | 4 yrs Data Scientist. Skills: Python, Machine Learning, Deep Learning, SQL. Tools: Git, Docker, AWS, PyTorch.",
    "Average Candidate": "John Smith | 2 yrs Data Analyst. Skills: Python, SQL, Data Viz. Tools: Tableau, Excel, Git.",
    "Weak Candidate": "Alice Johnson | 5 yrs Marketing Manager. Skills: SEO, Content Strategy, Copywriting. Tools: WordPress, Google Analytics."
}

print("\n--- Running Evaluation & Logging to LangSmith ---")
for candidate_type, resume_text in resumes.items():
    print(f"\nEvaluating: {candidate_type}")
    result = full_screening_pipeline.invoke({"resume": resume_text, "job_description": job_desc})
    print(f"Score: {result.get('score')}/100")
    print(f"Explanation: {result.get('explanation')}")

print("\nAll done! Check your LangSmith dashboard to grab screenshots of your trace logs.")

--- API Setup ---
Enter Google Gemini API Key: ··········
Enter LangSmith API Key: ··········

--- Building Resume Screening Pipeline ---

--- Running Evaluation & Logging to LangSmith ---

Evaluating: Strong Candidate
Score: 98/100
Explanation: The candidate is an excellent fit for the Data Scientist role. They exceed the experience requirement with 4 years (vs 3+). All required skills (Python, Machine Learning, SQL) are present, along with an additional relevant skill (Deep Learning). Similarly, all required tools (Git, Docker, AWS) are covered, with an extra relevant tool (PyTorch).

Evaluating: Average Candidate
Score: 50/100
Explanation: The candidate possesses foundational skills such as Python, SQL, and Git, which align with some job requirements. However, there are significant gaps: the candidate has 2 years of experience, falling short of the 3+ years required. Crucially, the candidate does not list Machine Learning, which is a core skill for a Data Scientist role and explicit